<a href="https://colab.research.google.com/github/infjlady/Bioinformatic-Analysis-of-DNMT1/blob/main/DNMT1_MSA_PHYLOGENETIC_TREE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython py3Dmol


In [ ]:
from Bio import Entrez
from Bio import SeqIO
from Bio import AlignIO
from Bio.Blast import NCBIWWW
from Bio.Blast import NCBIXML

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive"

print(os.listdir(base_path))

In [ ]:
path = "/content/drive/MyDrive/DNMT-SIDE-PRO"

print(os.listdir(path))

In [ ]:
sequence_path = "/content/drive/MyDrive/DNMT-SIDE-PRO/Sequences"

print(sequence_path)

In [ ]:
import os

print(os.listdir(sequence_path))

In [ ]:
!apt-get install clustalo -y

In [ ]:
from Bio import SeqIO
import os

sequence_path = "/content/drive/MyDrive/DNMT-SIDE-PRO/Sequences"

output_fasta = "/content/all_dnmt_sequences.fasta"

all_records = []

for file in os.listdir(sequence_path):
    if file.endswith(".fasta") or file.endswith(".fa"):

        file_path = os.path.join(sequence_path, file)

        records = list(SeqIO.parse(file_path, "fasta"))

        all_records.extend(records)

SeqIO.write(all_records, output_fasta, "fasta")

print(f"Combined {len(all_records)} sequences")
print(f"Saved to: {output_fasta}")

In [ ]:
!clustalo -i /content/all_dnmt_sequences.fasta \
-o /content/dnmt_alignment.aln \
--force

In [ ]:
with open("/content/dnmt_alignment.aln", "r") as file:
    alignment = file.read()

print(alignment[:5000])

In [ ]:
!pip install biopython matplotlib

In [ ]:
from Bio import AlignIO

alignment = AlignIO.read("/content/dnmt_alignment.aln", "fasta")

AlignIO.write(alignment, "/content/dnmt_alignment.phy", "phylip")

print("PHYLIP file created")

In [ ]:
from Bio import Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator
from Bio.Phylo.TreeConstruction import DistanceTreeConstructor
from Bio import AlignIO

# Read alignment
alignment = AlignIO.read("/content/dnmt_alignment.phy", "phylip")

# Calculate evolutionary distance
calculator = DistanceCalculator('identity')
distance_matrix = calculator.get_distance(alignment)

print(distance_matrix)

# Construct tree
constructor = DistanceTreeConstructor()
tree = constructor.nj(distance_matrix)

print(tree)

In [ ]:
import matplotlib.pyplot as plt
from Bio import Phylo

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1)

Phylo.draw(tree, axes=ax)

plt.show()

In [ ]:
from Bio import AlignIO

alignment = AlignIO.read("/content/dnmt_alignment.aln", "fasta")

alignment_length = alignment.get_alignment_length()

print("Alignment Length:", alignment_length)

In [ ]:
for i in range(alignment_length):

    column = alignment[:, i]

    unique_residues = set(column)

    if len(unique_residues) == 1 and "-" not in unique_residues:
        print(f"Position {i+1}: Conserved residue = {column[0]}")

In [ ]:
!pip install weblogo
!apt-get install ghostscript -y

In [ ]:
from Bio import AlignIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from Bio import SeqIO

alignment = AlignIO.read("/content/dnmt_alignment.aln", "fasta")

start = 1100
end = 1150

records = []

for record in alignment:

    region = record.seq[start:end]

    records.append(
        SeqRecord(
            Seq(str(region)),
            id=record.id,
            description=""
        )
    )

SeqIO.write(records, "/content/motif_region.fasta", "fasta")

print("Motif region FASTA created")

In [ ]:
!weblogo \
-F png \
-A protein \
-i 0 \
-o /content/dnmt_logo.png \
< /content/motif_region.fasta

In [ ]:
from IPython.display import Image

Image("/content/dnmt_logo.png")

In [ ]:
from Bio import AlignIO

alignment = AlignIO.read("/content/dnmt_alignment.aln", "fasta")

print("Alignment length:", alignment.get_alignment_length())

In [ ]:
for record in alignment:

    if "P26358" in record.id:

        print(record.id)
        print(record.seq[:300])

In [ ]:
mutation_positions = [1312, 1511, 1546]

for pos in mutation_positions:

    column = alignment[:, pos-1]

    print(f"\nPosition {pos}")
    print(column)

In [ ]:
!pip install py3Dmol

In [ ]:
!wget https://files.rcsb.org/download/4WXX.pdb

In [ ]:
import py3Dmol

pdb_file = open("4WXX.pdb", "r").read()

view = py3Dmol.view(width=900, height=600)

view.addModel(pdb_file, "pdb")

view.setStyle({"cartoon": {"color": "lightgray"}})

view.zoomTo()

view.show()

In [ ]:
mutations = [1312, 1511, 1546]

view = py3Dmol.view(width=900, height=600)

view.addModel(pdb_file, "pdb")

view.setStyle({"cartoon": {"color": "lightgray"}})

for residue in mutations:

    view.addStyle(
        {"resi": residue},
        {"stick": {"color": "red"}}
    )

view.zoomTo()

view.show()

In [ ]:
view = py3Dmol.view(width=900, height=600)

view.addModel(pdb_file, "pdb")

view.setStyle({"cartoon": {"color": "lightgray"}})

# conserved motif
view.addStyle(
    {"resi": "1220-1250"},
    {"cartoon": {"color": "blue"}}
)

# mutation residues
for residue in [1312, 1511, 1546]:

    view.addStyle(
        {"resi": residue},
        {"stick": {"color": "red"}}
    )

view.zoomTo()

view.show()

In [ ]:
from Bio.PDB import PDBParser

parser = PDBParser()

structure = parser.get_structure("DNMT1", "4WXX.pdb")

for model in structure:
    for chain in model:
        for residue in chain:

            if residue.id[0] != " ":
                print(residue)

In [ ]:
import py3Dmol

pdb_file = open("4WXX.pdb", "r").read()

view = py3Dmol.view(width=900, height=600)

view.addModel(pdb_file, "pdb")

# Protein cartoon
view.setStyle({"cartoon": {"color": "lightgray"}})

# Ligand
view.addStyle(
    {"resn": "SAH"},
    {"stick": {"color": "green"}}
)

view.zoomTo({"resn": "SAH"})

view.show()

In [ ]:
view = py3Dmol.view(width=900, height=600)

view.addModel(pdb_file, "pdb")

view.setStyle({"cartoon": {"color": "lightgray"}})

# ligand
view.addStyle(
    {"resn": "SAH"},
    {"stick": {"color": "green"}}
)

# nearby residues
view.addStyle(
    {"within": {"distance": 5, "sel": {"resn": "SAH"}}},
    {"stick": {"color": "orange"}}
)

view.zoomTo({"resn": "SAH"})

view.show()